# Notebook 03: Single Cell Integration

This analysis integrates the single cell results from each patient so they can be compared. Batch correction was applied via Harmony.

In [1]:
import sys
from pathlib import Path

# Dependencies are pinned in the repository .venv (see envs/single_cell.yaml);
# do not pip install here, just make sure the right kernel is selected.
expected = "endometriosis-spatial-immune-atlas-main/.venv"
if expected not in str(Path(sys.prefix).resolve()):
    raise RuntimeError(
        f"Wrong kernel: {sys.prefix}\n"
        "Select the 'endo-single-cell' kernel (repository .venv) and rerun."
    )
print(sys.version.split()[0], sys.prefix)


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [2]:
# -- Load libraries
from pathlib import Path

import numpy as np
import scanpy as sc
import scanpy.external as sce
import seaborn as sns
import harmonypy
from session_info2 import session_info

from matplotlib import pyplot as plt

import anndata as ad
ad.settings.allow_write_nullable_strings = True

In [3]:
# -- Mount drive
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
    BASE_DIR = Path("/content/drive/MyDrive/endometriosis-spatial-immune-atlas-main")
except ImportError:
    BASE_DIR = Path("/Users/elenaeyre/Desktop/Science/spatial_tx/endometriosis-spatial-immune-atlas-main")

In [4]:
# -- Paths
project_dir = BASE_DIR

dataset = "GSE179640"

interim_data_dir = (
    project_dir
    / "data"
    / "interim"
    / dataset
)

input_file = (
    interim_data_dir
    / "qc.h5ad"
)


integration_figures_dir = (
    project_dir
    / "figures"
    / dataset
    / "integration"
)

interim_data_dir.mkdir(
    parents=True,
    exist_ok=True,
)


integration_figures_dir.mkdir(
    parents=True,
    exist_ok=True,
)

In [5]:
# -- Set palettes
patient_colors_list = [
    sns.husl_palette(n_colors=1, h=h, s=0.9, l=0.65).as_hex()[0]
    for h in np.linspace(0.01, 0.70, 14)
]

# map to patient IDs
patient_ids = ["C01", "C02", "C03",
               "E01", "E02", "E03", "E04", "E05",
               "E06", "E07", "E08", "E09", "E10", "E11"]


patient_palette = dict(zip(patient_ids, patient_colors_list))

tissue_palette = {
    "Ctrl": "#7FA25C",
    "EuE":  "#028090",
    "EcP":  "#456990",
    "EcO":  "#CE7DA5"
}



In [6]:
# -- Load QC object
combined = sc.read_h5ad(input_file)
print(combined)

print(f"Starting cells: {combined.n_obs}")
print(f"Starting genes: {combined.n_vars}")

AnnData object with n_obs × n_vars = 94891 × 30907
    obs: 'sample_id', 'patient_id', 'tissue_type', 'condition', 'lesion_site', 'dataset', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'total_counts_ribos', 'pct_counts_ribos', 'total_counts_hemos', 'pct_counts_hemos', 'n_genes', 'n_counts', 'outlier_mt', 'doublet_score', 'predicted_doublet'
    var: 'hemos', 'ribos', 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'n_cells'
    uns: 'scrublet'
    layers: 'counts'
Starting cells: 94891
Starting genes: 30907


In [7]:
# -- Normalize counts
sc.pp.normalize_total(combined)
sc.pp.log1p(combined)

In [8]:
# -- Select HVGs
sc.pp.highly_variable_genes(combined, n_top_genes=2000, batch_key="sample_id")

print(
    "Highly variable genes: "
    f"{combined.var['highly_variable'].sum()}"
)


# -- Plot highly variable genes
sc.pl.highly_variable_genes(combined, show = False)
plt.savefig(
    integration_figures_dir / "03_integration_hvgs.png",
    bbox_inches='tight',
    dpi=300
)
plt.close()

/Users/elenaeyre/Desktop/Science/Elena_Python_scripts/KG/venv/lib/python3.14/site-packages/scanpy/preprocessing/_highly_variable_genes.py:343: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  df["mean_bin"] = _get_mean_bins(df["means"], flavor,

Highly variable genes: 2000


In [9]:
# -- Dimensionality Reduction
# PCA uses HVGs, but all genes remain in the AnnData object
sc.tl.pca(
    combined,
    mask_var="highly_variable",
)

sc.pl.pca_variance_ratio(combined,
                         n_pcs=30,
                         log=True,
                         show = False)

plt.savefig(
    integration_figures_dir /
    "03_integration_variance_ratios.png",
    bbox_inches='tight',
    dpi=300
)
plt.close()

In [10]:
# -- Inspect uncorrected PCA
sc.pl.pca(
    combined,
    color=["patient_id", "patient_id", "pct_counts_mt", "pct_counts_mt"],
    dimensions=[
        (0, 1),
         (2, 3),
          (0, 1),
           (2, 3)
           ],
    ncols=2,
    size=2,
    title = [
        "UMAP by Patient ID - PC1 v PC2",
        "UMAP by Patient ID - PC3 v PC4",
        "UMAP by MT% - PC1 v PC2",
        "UMAP by MT% - PC3 v PC4"],
    show=False
)

plt.savefig(
    integration_figures_dir
    / "03_integration_pca_diagnostics.png",
    bbox_inches="tight",
    dpi=300,
)

plt.close()

In [11]:
# -- Harmony integration
ho = harmonypy.run_harmony(
    combined.obsm["X_pca"],
    combined.obs,
    "sample_id"
)

combined.obsm["X_pca_harmony"] = ho.Z_corr

2026-10-11 14:21:24,369 - harmonypy - INFO - Running Harmony
2026-10-11 14:21:24,370 - harmonypy - INFO -   Parameters:
2026-10-11 14:21:24,370 - harmonypy - INFO -     max_iter_harmony: 10
2026-10-11 14:21:24,371 - harmonypy - INFO -     max_iter_kmeans: 4
2026-10-11 14:21:24,371 - harmonypy - INFO -     epsilon_cluster: 0.001
2026-10-11 14:21:24,372 - harmonypy - INFO -     epsilon_harmony: 0.01
2026-10-11 14:21:24,372 - harmonypy - INFO -     nclust: 100
2026-10-11 14:21:24,372 - harmonypy - INFO -     block_size: 0.05
2026-10-11 14:21:24,372 - harmonypy - INFO -     lamb: dynamic (alpha=0.2)
2026-10-11 14:21:24,373 - harmonypy - INFO -     theta: [2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2.]
2026-10-11 14:21:24,373 - harmonypy - INFO -     sigma: [0.1 0.1 0.1 0.1 0.1]...
2026-10-11 14:21:24,373 - harmonypy - INFO -     verbose: True
2026-10-11 14:21:24,374 - harmonypy - INFO -     random_state: 0
2026-10-11 14:21:24,374 - harmonypy - INFO -   Data: 50 PC

In [ ]:
venv: endometriosis-spatial-immune-atlas

In [12]:
## -- Nearest Neighbors
sc.pp.neighbors(
    combined,
    use_rep="X_pca_harmony",
    n_neighbors=15
)

KeyboardInterrupt: 

In [ ]:
# -- UMAP
sc.tl.umap(combined)

In [ ]:
# -- UMAP by patient
sc.pl.umap(
    combined,
    color="patient_id",
    palette=patient_palette,
    size = 4,
    show=False,
    title = "UMAP by Patient ID"
)

plt.savefig(
    integration_figures_dir / "03_integration_umap_patient_id.png",
    bbox_inches='tight',
    dpi=300
)
plt.close()



sc.pl.umap(
    combined,
    color="tissue_type",
    palette=tissue_palette,
    size = 4,
    show=False,
    title = "UMAP by Tissue Type"
)

plt.savefig(
    integration_figures_dir / "03_integration_umap_tissue_type.png",
    bbox_inches='tight',
    dpi=300
)
plt.close()

In [ ]:
# -- Save integrated object
output_file = (
    interim_data_dir
    / "integrated.h5ad"
)

combined.write_h5ad(
    output_file
)


print("\nIntegration complete.")
print(f"Final cells: {combined.n_obs}")
print(f"Genes retained: {combined.n_vars}")
print(
    "HVGs used for PCA: "
    f"{combined.var['highly_variable'].sum()}"
)
print(f"Saved integrated object to:\n{output_file}")

In [ ]:
#-- Session info
info = session_info()
print(info)